<a href="https://colab.research.google.com/github/srroane/PublicInfrastructureProjectsBR/blob/main/Public_Infrastructure_Projects_Location_Access_Change.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Job-access change at displacement sites (IPEA Access to Opportunities, 2019)
# Runs on Google Colab (colab.research.google.com): paste this whole script into
# one cell and press Run. It installs what it needs and, if the two IPEA files
# are not there yet, asks you to upload them.
# Coordinates are from Google Maps (right-click a place -> copy "lat, lng"),
import math, os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h3>=4", "pandas"], check=True)
import h3          # Uber H3, version 4.x
import pandas as pd

ACCESS_FILE = "aop_access_publictransport_2019_v2.csv"
POP_FILE = "aop_population_2010_v2.csv"

# name: (latitude, longitude, city code in the IPEA files)
SITES = {
    "Jardim Edite":      (-23.6131502, -46.6940215, "spo"),
    "Vila Autodromo":    (-22.9731053, -43.3939562, "rio"),
    "Parque Carioca":    (-22.9649556, -43.4095416, "rio"),
    "Providencia":       (-22.9013615, -43.1952020, "rio"),
    "Nabuco de Freitas": (-22.9045133, -43.2004043, "rio"),
}
# (origin, destination) pairs to compare
PAIRS = [("Vila Autodromo", "Parque Carioca"), ("Providencia", "Nabuco de Freitas")]

missing = [f for f in (ACCESS_FILE, POP_FILE) if not os.path.exists(f)]
if missing:
    print("Please upload:", ", ".join(missing))
    from google.colab import files   # only works on Colab
    files.upload()

acc = pd.read_csv(ACCESS_FILE, usecols=["id_hex", "abbrev_muni", "mode", "peak", "CMATT60"])
acc = acc[(acc.peak == 1) & (acc["mode"] == "public_transport")]
pop = pd.read_csv(POP_FILE, usecols=["id_hex", "abbrev_muni", "P001"])

def km(lat1, lon1, lat2, lon2):  # straight-line (haversine) distance
    p = math.radians
    x = (math.sin(p(lat2 - lat1) / 2) ** 2
         + math.cos(p(lat1)) * math.cos(p(lat2)) * math.sin(p(lon2 - lon1) / 2) ** 2)
    return 2 * 6371 * math.asin(math.sqrt(x))

def measures(lat, lng, city):
    a = acc[acc.abbrev_muni == city].set_index("id_hex")["CMATT60"]
    p = pop[pop.abbrev_muni == city].set_index("id_hex")["P001"]
    hx = h3.latlng_to_cell(lat, lng, 9)
    within = lambda r: [c for c in h3.grid_disk(hx, 8) if km(lat, lng, *h3.cell_to_latlng(c)) <= r]
    single = a.get(hx)
    df = pd.DataFrame({"acc": a}).join(p.rename("pop")).fillna({"pop": 0})
    return {
        "hexagon": hx,
        "single": single,
        "ring1_median (500 m, used in text)": a.reindex(list(h3.grid_disk(hx, 1))).median(),
        "ring2_median": a.reindex(list(h3.grid_disk(hx, 2))).median(),
        "within_500m_median": a.reindex(within(0.5)).median(),
        "within_1km_median": a.reindex(within(1.0)).median(),
        "pct_unweighted": round((a < single).mean() * 100, 1),
        "pct_pop_weighted": round(df.loc[df.acc < single, "pop"].sum() / df["pop"].sum() * 100, 1),
    }

res = {n: measures(*v) for n, v in SITES.items()}
print(pd.DataFrame(res).T.to_string())

for o, d in PAIRS:
    print(f"\n{o} -> {d}: {km(*SITES[o][:2], *SITES[d][:2]):.2f} km")
    for k in ["single", "ring1_median (500 m, used in text)", "ring2_median",
              "within_500m_median", "within_1km_median"]:
        x, y = res[o][k], res[d][k]
        print(f"  {k:36s} {x:>10,.0f} -> {y:>10,.0f}  ({y / x - 1:+.1%})")

                           hexagon     single ring1_median (500 m, used in text) ring2_median within_500m_median within_1km_median pct_unweighted pct_pop_weighted
Jardim Edite       89a81005213ffff  1773681.0                          1640235.0    1619213.0          1640235.0         1640235.0           94.5             91.6
Vila Autodromo     89a8a075393ffff   180125.0                           138906.0     138906.0           138906.0          138906.0           81.3             51.8
Parque Carioca     89a8a075163ffff   140076.0                            94234.0      65728.0            94234.0           60693.0           77.1             44.5
Providencia        89a8a06a19bffff   413271.0                           680434.0     663825.0           684940.0          737119.0           91.0             74.1
Nabuco de Freitas  89a8a06a52bffff   663825.0                           663825.0     799419.0           608007.0          769943.5           97.2             90.7

Vila Autodromo -> Par